<a href="https://colab.research.google.com/github/gausul9365/BUILDING-ML/blob/main/Supervised_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STeps Applying TRL and SFT


1.   Load Model and tokenizer
2.   load dataset


1.   Configure LoRA
2.   Configure Training
3. create trainer
4. train
5. Save Adapter






In [1]:
pip install -U torch transformers datasets peft trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 45.8 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 46.1 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 63.2 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 23.4 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 51.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 89.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 75.0 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 77.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 51.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 30.8 MB/s  0:00:08
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 105.0 MB/s  0:00:00

In [3]:
!pip uninstall -y torchaudio torchvision

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train[:1%]"
)

In [3]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
!pip install -U trl

In [13]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./lora-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    max_length=256,
    dataset_text_field="text",
    report_to="none",
    fp16=True,
    bf16=False
)

In [14]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:342: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Truncating train dataset:   0%|          | 0/21197 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/21197 [00:00<?, ? examples/s]

In [15]:
trainer.train()

Step,Training Loss
10,1.932071
20,1.757224
30,1.762893
40,1.716264
50,1.744878
60,1.766044
70,1.694018
80,1.714530
90,1.681368
100,1.685303


TrainOutput(global_step=100, training_loss=1.7454593467712403, metrics={'train_runtime': 120.3262, 'train_samples_per_second': 6.649, 'train_steps_per_second': 0.831, 'total_flos': 371027154094080.0, 'train_loss': 1.7454593467712403, 'epoch': 0.03773940937824323})

In [16]:
trainer.model.save_pretrained("./my-lora-adapter")

tokenizer.save_pretrained("./my-lora-adapter")

('./my-lora-adapter/tokenizer_config.json',
 './my-lora-adapter/chat_template.jinja',
 './my-lora-adapter/tokenizer.json')

 **testing it**

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./my-lora-adapter"

# 1. Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto",
)

# 2. Attach your trained LoRA adapter
ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()

# 3. Test it with a story prompt
prompt = "my name is Gausul not max"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

my name is Gausul not maxwell

Gauss was a mathematician who lived in the 19th century. He made important discoveries about numbers and shapes, which are still used today. His work helped people understand more about mathematics and how it works. Just like Maxwell, Gauss loved to make new things happen with his mind! Does that help explain it? Let me know if you have any other questions. I'm happy to answer them too! How can we help you today? Please feel free to ask us anything you need help with. Goodbye! (I hope this helps!)



> **Observation:**


I fine-tuned a LoRA adapter (`r=8`, 100 steps) on the `TinyStories` dataset on top of `Qwen2.5-0.5B-Instruct`.

When prompted with a storybook starter like

*"Once upon a time, there was a little dog named Max..."*,

the model followed the `TinyStories`
 style and generated a coherent children's story. However,

 when given an out-of-distribution prompt

 (*"my name is Gausul not max"*),

 it fell back to the frozen base model's pretraining knowledge—auto-completing *"max"* to *"maxwell"* and associating *"Gausul"* with the 19th-century mathematician Gauss. This shows that a lightweight LoRA adapter nudges the model toward the fine-tuned domain without erasing the base model's general world knowledge.